In [73]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
import polars as pl
import pandas as pd

# Assuming 'df_full' is your original unsplit dataset
df_full = pl.read_csv("/home/biostat/Register_data/data_request/RA/Stratafit/05_data_ready_to_use/mockClientData/realSplitData/*.csv").to_pandas()

columns = ['DAS28', 'CRP', 'ESR', 'SJC28', 'TJC28']

central_imputer = IterativeImputer(estimator=BayesianRidge(), max_iter=20, random_state=42)
df_central_imputed = pd.DataFrame(
    central_imputer.fit_transform(df_full[columns]), 
    columns=columns
)

In [74]:
import pandas as pd
import numpy as np
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge

# def apply_federated_mice(df: pd.DataFrame, model_config: dict):
#     """
#     Applies the federated MICE parameters to a new/full dataframe.
#     """
#     # 1. Extract info from the Vantage6 result
#     state = model_config['state']
#     columns = model_config['parameters']['columns']
#     global_estimates = state['global_estimates']
    
#     # Ensure we only work with the columns the model was trained on
#     data = df[columns].copy().values
    
#     # 2. Create a 'blank' imputer
#     # We set max_iter=1 because we only need the imputer to perform 
#     # the transformation logic once using our injected weights.
#     imputer = IterativeImputer(estimator=BayesianRidge(), max_iter=1, random_state=42)
    
#     # 3. 'Prime' the imputer
#     # It needs to 'fit' once just to build the internal imputation_sequence_ 
#     # (the list of regression models for each column).
#     imputer.fit(data)
    
#     # 4. Surgical Injection of Federated Weights
#     for i, est_data in enumerate(global_estimates):
#         # Access the internal regression model for this step in the chain
#         # Using index [2] to get the estimator object
#         target_estimator = imputer.imputation_sequence_[i][2]
        
#         # Manually set the learned coefficients and intercept
#         target_estimator.coef_ = np.array(est_data['coef'])
#         target_estimator.intercept_ = est_data['intercept']
        
#     # 5. Transform (the actual imputation)
#     imputed_values = imputer.transform(data)
    
#     # 6. Return as a nice DataFrame
#     return pd.DataFrame(imputed_values, columns=columns, index=df.index)

# --- EXAMPLE USAGE ---
# vantage_output = [{'schema_version': 1, ...}] # Your JSON result
# df_full = pd.read_csv("your_data.csv")

# df_imputed = apply_federated_mice(df_full, vantage_output[0])
# print(df_imputed.head())

def apply_federated_mice(df: pd.DataFrame, model_config: dict):
    state = model_config['state']
    params = model_config['parameters']
    columns = params['columns']
    global_estimates = state['global_estimates']
    
    # Use the same max_iter as the training/central run
    max_iter = 1 #params.get('max_iter', 20)
    
    data = df[columns].copy().values
    
    # 1. Initialize with the same parameters as central
    # We must match max_iter to ensure the chain converges similarly
    imputer = IterativeImputer(estimator=BayesianRidge(), max_iter=max_iter, random_state=42)
    
    # 2. Prime the imputer
    imputer.fit(data)
    
    # 3. Corrected Weight Injection: Match by feature index
    # Create a lookup map for our federated estimates
    fed_map = {est['feat_idx']: est for est in global_estimates}
    
    for i in range(len(imputer.imputation_sequence_)):
        # The first element of the sequence tuple is the target feature index
        target_feat_idx = imputer.imputation_sequence_[i][0]
        target_estimator = imputer.imputation_sequence_[i][2]
        
        if target_feat_idx in fed_map:
            est_data = fed_map[target_feat_idx]
            target_estimator.coef_ = np.array(est_data['coef'])
            target_estimator.intercept_ = est_data['intercept']
            # BayesianRidge specifically needs this flag set to True
            target_estimator.fitted_ = True 
    
    # 4. Transform
    imputed_values = imputer.transform(data)
    
    return pd.DataFrame(imputed_values, columns=columns, index=df.index)


In [75]:
model_config = {'schema_version': 1, 'type': 'imputation', 'strategy': 'mice', 'fitted': True, 'parameters': 
                {'columns': ['DAS28', 'CRP', 'ESR', 'SJC28', 'TJC28'], 'max_iter': 20}, 'state': {'initial_means': {'DAS28': 3.5231639094471046, 'CRP': 0.8559714620797497, 'ESR': 25.372279987136885, 'SJC28': 2.1992859758330283, 'TJC28': 3.8378180486911955}, 'global_estimates': [
                    {'feat_idx': 0, 'neighbor_indices': [1, 2, 3, 4], 'coef': [-0.02083034096564324, 0.027649178633027016, 0.11322051242906457, 0.15171397243433316], 'intercept': 2.033759460692197}, 
                    {'feat_idx': 1, 'neighbor_indices': [0, 2, 3, 4], 'coef': [-0.37011400993139354, 0.053574374300167094, 0.1373221351286292, 0.06086007999041301], 'intercept': 0.2638980488421181}, 
                    {'feat_idx': 2, 'neighbor_indices': [0, 1, 3, 4], 'coef': [14.92225155036105, 4.049840446788246, -1.504184288002255, -2.279132855745287], 'intercept': -18.850442971279513}, 
                    {'feat_idx': 3, 'neighbor_indices': [0, 1, 2, 4], 'coef': [2.219225600448694, 0.37951186323227576, -0.05571737137706925, -0.21047484130233865], 'intercept': -3.770855003572196}, 
                    {'feat_idx': 4, 'neighbor_indices': [0, 1, 2, 3], 'coef': [4.782573949734026, 0.27411220569618977, -0.1359996227758956, -0.33905177370698686], 'intercept': -9.153912455253117}]}, 'metadata': {'created_at': '2026-03-11T08:25:28.783908+00:00Z', 'n_organizations': 3}}

In [76]:
# Assuming 'central_imputer' is your fitted sklearn IterativeImputer
# and 'columns' is your list ['DAS28', 'CRP', 'ESR', 'SJC28', 'TJC28']

# central_results = []
# for step in central_imputer.imputation_sequence_:
#     feat_idx = step[0]
#     neighbor_indices = step[1]
#     estimator = step[2]
    
#     if int(feat_idx == 3):
#         central_results.append({
#             "feat_idx": int(feat_idx),
#             "column_name": columns[feat_idx],
#             "neighbor_indices": neighbor_indices.tolist(),
#             "coef": estimator.coef_.tolist(),
#             "intercept": float(estimator.intercept_)
#         })

# # Print them out to compare side-by-side
# import json
# print(json.dumps(central_results, indent=2))

In [77]:
# In your test script
print(central_imputer.imputation_sequence_)

[_ImputerTriplet(feat_idx=np.int64(4), neighbor_feat_idx=array([0, 1, 2, 3]), estimator=BayesianRidge()), _ImputerTriplet(feat_idx=np.int64(3), neighbor_feat_idx=array([0, 1, 2, 4]), estimator=BayesianRidge()), _ImputerTriplet(feat_idx=np.int64(1), neighbor_feat_idx=array([0, 2, 3, 4]), estimator=BayesianRidge()), _ImputerTriplet(feat_idx=np.int64(2), neighbor_feat_idx=array([0, 1, 3, 4]), estimator=BayesianRidge()), _ImputerTriplet(feat_idx=np.int64(0), neighbor_feat_idx=array([1, 2, 3, 4]), estimator=BayesianRidge()), _ImputerTriplet(feat_idx=np.int64(4), neighbor_feat_idx=array([0, 1, 2, 3]), estimator=BayesianRidge()), _ImputerTriplet(feat_idx=np.int64(3), neighbor_feat_idx=array([0, 1, 2, 4]), estimator=BayesianRidge()), _ImputerTriplet(feat_idx=np.int64(1), neighbor_feat_idx=array([0, 2, 3, 4]), estimator=BayesianRidge()), _ImputerTriplet(feat_idx=np.int64(2), neighbor_feat_idx=array([0, 1, 3, 4]), estimator=BayesianRidge()), _ImputerTriplet(feat_idx=np.int64(0), neighbor_feat_id

In [78]:
df_fed_combined = apply_federated_mice(df=df_full, model_config=model_config)

/home/josef/stratafit_repos/stratafit-org-github/strata-fit-v6-imputation-py/.venv/lib/python3.12/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


In [79]:
import numpy as np

# Combine your imputed node dataframes into one: df_fed_combined
diff = df_central_imputed - df_fed_combined
mse = np.mean(diff.values**2)

print(f"Federated vs Centralized MSE: {mse}")

Federated vs Centralized MSE: 11.276609933019794


In [72]:
for col in columns:
    col_mse = np.mean((df_central_imputed[col] - df_fed_combined[col])**2)
    print(f"MSE for {col}: {col_mse}")

MSE for DAS28: 9.73705505615971e-05
MSE for CRP: 0.0001362982657472478
MSE for ESR: 0.00023035727025053085
MSE for SJC28: 0.00025650813340704655
MSE for TJC28: 0.0028183028388556793


In [63]:
def compare_final_weights(central_imputer, fed_config):
    fed_map = {est['feat_idx']: est for est in fed_config['state']['global_estimates']}
    columns = fed_config['parameters']['columns']
    n_cols = len(columns)
    
    # Take only the LAST iteration's worth of estimators
    # This slices the last 'n_cols' from the sequence
    final_sequence = central_imputer.imputation_sequence_[-n_cols:]
    
    comparison = []
    
    for target_idx, _, estimator in final_sequence:
        fed_est = fed_map[target_idx]
        
        # Calculate the difference
        coef_diff = np.linalg.norm(estimator.coef_ - np.array(fed_est['coef']))
        intercept_diff = abs(estimator.intercept_ - fed_est['intercept'])
        
        comparison.append({
            "Column": columns[target_idx],
            "Central_Intercept": estimator.intercept_,
            "Fed_Intercept": fed_est['intercept'],
            "Coef_L2_Diff": coef_diff,
            "Intercept_Diff": intercept_diff
        })
        
    return pd.DataFrame(comparison)

final_audit = compare_final_weights(central_imputer, model_config)
print(final_audit)

  Column  Central_Intercept  Fed_Intercept  Coef_L2_Diff  Intercept_Diff
0  TJC28          -9.187400      -9.153912      0.156259        0.033487
1  SJC28          -3.811036      -3.770855      0.076327        0.040181
2    CRP          -0.307807       0.263898      0.248897        0.571705
3    ESR         -18.817740     -18.850443      0.012871        0.032703
4  DAS28           2.033981       2.033759      0.000398        0.000222
